# Open-System TEBD for the Transverse-Field Ising Model
## Vectorized Density Matrix MPS with Lindblad Dissipation

## Abstract

This notebook demonstrates **open-system TEBD** applied to the TFIM in contact with a Markovian environment. While closed systems evolve via the Schrödinger equation, open systems obey the **Lindblad master equation**:

$$
\frac{d\rho}{dt} = -i[H, \rho] + \sum_k \gamma_k \left( L_k \rho L_k^\dagger - \frac{1}{2}\{L_k^\dagger L_k, \rho\} \right)
$$

where $\rho$ is the density matrix, $L_k$ are **Lindblad jump operators** (encoding the system-bath coupling), and $\gamma_k$ are their rates.

### Strategy: Vectorized Density Matrix MPS

Rather than storing $\rho$ as a $d^L \times d^L$ matrix, we **vectorize** it into a ket:
$$
|\rho\rangle\rangle = \sum_{m,n} \rho_{mn} |m\rangle |n\rangle
$$
and represent $|\rho\rangle\rangle$ as an MPS with **super-site** local dimension $d_s = d^2$. The Lindblad equation becomes a linear ODE for $|\rho\rangle\rangle$, which we evolve with the same TEBD machinery as the closed system, but using the Lindblad **superoperator** $K$ as the generator.

This notebook uses `a_mps.py`, `b_model.py` (`TFIModelOpen`), and `TFIM_tebd_open.py`.

## Vectorized Density Matrix

### Super-site Representation

For a physical spin with $d=2$ states $\{|0\rangle, |1\rangle\}$, we introduce a **super-site** with local dimension $d_s = d^2 = 4$. The vectorized density matrix at site $i$ is:

$$
|\rho\rangle\rangle = \sum_{m,n=0}^{d-1} \rho_{mn} |m\rangle_{\text{ket}} \otimes |n\rangle_{\text{bra}}
$$

We use the **super-index** $\alpha = m \cdot d + n \in \{0, 1, 2, 3\}$:

| $\alpha$ | $(m, n)$ | Physical meaning |
|---------|---------|------------------|
| 0 | $(0,0)$ | $\rho_{00}$ (population $|0\rangle$) |
| 1 | $(0,1)$ | $\rho_{01}$ (coherence) |
| 2 | $(1,0)$ | $\rho_{10}$ (coherence) |
| 3 | $(1,1)$ | $\rho_{11}$ (population $|1\rangle$) |

### MPS for the Vectorized Density Matrix

The full vectorized density matrix is written as an MPS over $L$ super-sites:

$$
|\rho\rangle\rangle = \sum_{\alpha_0, \ldots, \alpha_{L-1}} \Lambda^{[0]} B^{[0]}_{\alpha_0} \cdots B^{[L-1]}_{\alpha_{L-1}} |\alpha_0 \cdots \alpha_{L-1}\rangle
$$

Each $B^{[i]}$ is a rank-3 tensor with legs `(chi_L, ds, chi_R)`, exactly as in the closed-system case, but with $d_s = 4$ instead of $d = 2$.

In [ ]:
import numpy as np
from scipy.linalg import expm, svd as scipy_svd
import matplotlib.pyplot as plt
import matplotlib as mpl

from a_mps import SimpleMPS, init_purification_MPS, split_truncate_theta
from b_model import TFIModelOpen

## The Lindblad Superoperator

The Lindblad master equation acts linearly on $|\rho\rangle\rangle$ via the superoperator $K = K_{\text{coh}} + K_{\text{diss}}$:

### Coherent part
$$
K_{\text{coh}} = -i\bigl(H \otimes I - I \otimes H^*\bigr)
$$
where $H$ acts on the ket space and $H^*$ on the bra space.

### Dissipative part
For each jump operator $L$ with rate $\gamma$:
$$
K_{\text{diss}} = \gamma \left( L \otimes L^* - \frac{1}{2} L^\dagger L \otimes I - \frac{1}{2} I \otimes (L^\dagger L)^T \right)
$$

Here we use **local dephasing** as the jump operator: $L_i = \sigma^z_i$ with rate $\gamma$. This choice causes exponential decay of off-diagonal elements:
$$
\rho_{01}(t) = \rho_{01}(0)\, e^{-2\gamma t}
$$
which provides a clean analytical benchmark.

### Bond Superoperator

Just like the Hamiltonian is split into bond terms $H = \sum_i h_{i,i+1}$, the superoperator is split into bond generators $K = \sum_i K_{i,i+1}$, each acting on the super-sites $i$ and $i+1$. The bond generators are stored as rank-4 tensors of shape `(ds, ds, ds, ds)` in `TFIModelOpen.K_bonds`.

The **index ordering** in `K_bonds` follows the super-site (interleaved ket-bra) convention:
$$
K_{\alpha_i' \alpha_{i+1}',\, \alpha_i \alpha_{i+1}} \quad \text{where} \quad \alpha_k = m_k d + n_k
$$

In [ ]:
# TFIModelOpen class (from b_model.py)
# Builds K_bonds — the Lindblad generator decomposed into two-super-site bond operators.

# class TFIModelOpen:
#     """Open TFIM: H = -J Σ σ^x_i σ^x_{i+1} + B Σ σ^z_i with Lindblad jump operators.
#
#     Builds the Lindblad generator K_bonds for the vectorized density matrix MPS.
#     Each K_bond acts on super-sites (i, i+1) with super-index α = m*d + n (ds = d²).
#     """
#
#     def __init__(self, L, J, B, jump_ops, gammas, bc='finite'):
#         ...
#         self._init_K_bonds()
#
#     def _init_K_bonds(self):
#         # For each bond (i, i+1):
#         # 1. Build two-site Hamiltonian H_bond in (ket) space
#         # 2. Build coherent superoperator: K_coh = -i(H_bond ⊗ I - I ⊗ H_bond*)
#         # 3. For each jump op L: add dissipator D[L] = L⊗L* - ½ L†L⊗I - ½ I⊗(L†L)^T
#         # 4. Convert from (ket,bra)-flat to super-site (interleaved) index ordering
#         # 5. Store as rank-4 tensor K_bonds[i] of shape (ds, ds, ds, ds)
#
#     K_bonds : list of np.ndarray, shape (ds, ds, ds, ds)

# Demonstrate TFIModelOpen construction
L_demo = 4
d = 2
ds = d * d
sz = np.array([[1., 0.], [0., -1.]])

M_open = TFIModelOpen(L=L_demo, J=1.0, B=0.5,
                      jump_ops=[sz], gammas=[0.3], bc='finite')
print("Number of K_bonds:", len(M_open.K_bonds))
print("Shape of K_bonds[0]:", M_open.K_bonds[0].shape,
      "  (ds, ds, ds, ds) with ds =", ds)
K0_mat = M_open.K_bonds[0].reshape(ds * ds, ds * ds)
eigvals = np.linalg.eigvals(K0_mat)
print("Max real part of K_bond[0] eigenvalues (should be ≤ 0):",
      np.max(np.real(eigvals)).round(6))

## Purification MPS Initialization

We initialize the density matrix as a pure product state $\rho_0 = |+x\rangle\langle+x|^{\otimes L}$ where $|{+x}\rangle = (|0\rangle + |1\rangle)/\sqrt{2}$.

In the vectorized representation, this corresponds to:
$$
|\rho_0\rangle\rangle = \bigotimes_{i=0}^{L-1} |v_i\rangle\rangle
$$
where the local super-site vector has components:
$$
(v_i)_\alpha = \rho^{(i)}_{mn} = \frac{1}{2} \quad \forall\, m, n \in \{0, 1\}
$$

The `init_purification_MPS` function in `a_mps.py` builds this MPS directly:
- Each $B^{[i]}$ has shape `(1, ds, 1)` (product state, $\chi=1$)
- Each $\Lambda^{[i]} = [1.0]$ (no entanglement initially)

Other initial states available: `'up'` ($\rho = |0\rangle\langle 0|^{\otimes L}$) and `'mixed'` ($\rho = (I/d)^{\otimes L}$).

In [ ]:
# init_purification_MPS (from a_mps.py)
# Returns a product-state MPS representing the vectorized density matrix.

# def init_purification_MPS(L, d, bc='finite', state='+x'):
#     """Return a product-state purification MPS for the vectorized density matrix.
#
#     Each super-site has local dimension ds = d², with super-index α = m*d + n
#     where m is the ket index and n is the bra index.
#
#     state : '+x'    → ρ = |+x⟩⟨+x|⊗L,  |+x⟩ = (|0⟩+|1⟩)/√2
#             'up'    → ρ = |0⟩⟨0|⊗L
#             'mixed' → ρ = (I/d)⊗L (maximally mixed), Frobenius-normalised
#     """
#     ds = d * d
#     if state == '+x':
#         v = np.full(ds, 0.5)            # rho_{mn} = 1/2 for all m,n
#     elif state == 'up':
#         v = np.zeros(ds)
#         v[0] = 1.0                       # rho_{00} = 1
#     elif state == 'mixed':
#         v = np.zeros(ds)
#         for n in range(d):
#             v[n * d + n] = 1.0 / d
#         v /= np.linalg.norm(v)           # Frobenius normalise
#     B = np.zeros([1, ds, 1], dtype=complex)
#     B[0, :, 0] = v
#     S = np.ones([1])
#     Bs = [B.copy() for _ in range(L)]
#     Ss = [S.copy() for _ in range(L)]
#     return SimpleMPS(Bs, Ss, bc=bc)

# Demonstrate initialization
L_demo = 6
d = 2
psi_rho = init_purification_MPS(L=L_demo, d=d, bc='finite', state='+x')
print("Super-site local dimension ds =", psi_rho.Bs[0].shape[1])
print("Initial bond dimensions:", psi_rho.get_chi())
print("Local vector at site 0 (should all be 0.5):", psi_rho.Bs[0][0, :, 0])
print("Shape of B[0]:", psi_rho.Bs[0].shape, "  (chi_L=1, ds=4, chi_R=1)")

## TEBD with Norm Tracking

### Why the MPS norm changes

Unlike unitary evolution (where $\|\psi\|=1$ is conserved), the Lindblad superoperator $K$ is **non-Hermitian**. The TEBD gates $U = e^{K \Delta t}$ are not unitary, so after each bond update the MPS norm $\||\rho\rangle\rangle\|$ changes.

During truncation in `split_truncate_theta`, we **renormalize** the Schmidt values: $S \to S/\|S\|$. This keeps the MPS Frobenius-normalized ($\||\rho\rangle\rangle\|_F = 1$) but discards the overall norm factor.

### Log-norm accumulation

We track the **log of the cumulative norm factor** divided out at each step:
$$
\log\mathcal{N}(t) = \sum_{\text{steps}} \log\|S_{\text{kept}}\|
$$

The trace of the physical density matrix is then:
$$
\text{Tr}(\rho(t)) = e^{\log\mathcal{N}(t)} \cdot \langle\langle I | \tilde{\rho}(t) \rangle\rangle
$$
where $|\tilde{\rho}\rangle\rangle$ is the Frobenius-normalized MPS and $\langle\langle I|$ is the identity bra vector (see below). This must equal 1 as a sanity check.

### `accum_norm` function

Each call to `accum_norm` runs one full TEBD step (even-odd-even), accumulates `log_norm`, and returns the maximum truncation error over all bond updates.

In [ ]:
# update_bond and accum_norm (from TFIM_tebd_open.py)

def calc_U_bonds_open(K_bonds, dt):
    """Exponentiate Lindblad generators into 2nd-order Trotter TEBD gates.

    Even-index bonds: exp(dt/2 * K)  ->  half-step
    Odd-index  bonds: exp(dt   * K)  ->  full-step
    Applied as [even, odd, even] per TEBD step -> 2nd-order accuracy.
    """
    U_bonds = {}
    for i, K in enumerate(K_bonds):
        ds = K.shape[0]
        K_mat = K.reshape(ds * ds, ds * ds)
        step = dt / 2 if i % 2 == 0 else dt
        U_bonds[i] = expm(step * K_mat).reshape(ds, ds, ds, ds)
    return U_bonds


def update_bond(psi, i, U_bond, chi_max, eps):
    """Apply U_bond on super-sites i and i+1, truncate, and return norm info.

    Returns
    -------
    log_norm  : float  -- log of the SVD norm factor divided out
    trunc_err : float  -- 1 - ||Y_kept||^2 / ||Y_full||^2
    """
    j = (i + 1) % psi.L
    theta  = psi.get_theta2(i)
    Utheta = np.tensordot(U_bond, theta, axes=([2, 3], [1, 2]))
    Utheta = np.transpose(Utheta, [2, 0, 1, 3])
    chivL, dL, dR, chivR = Utheta.shape
    _, Y, _ = scipy_svd(Utheta.reshape(chivL * dL, dR * chivR), full_matrices=False)
    norm_old = np.sum(np.square(Y))
    Ai, Sj, Bj, _ = split_truncate_theta(Utheta, chi_max, eps)
    norm_new = np.sum(np.square(np.sort(Y)[::-1][:Sj.shape[0]]))
    log_norm  = np.log(np.sqrt(norm_new))
    trunc_err = 1.0 - norm_new / norm_old
    Gi = np.tensordot(np.diag(psi.Ss[i] ** (-1)), Ai, axes=[1, 0])
    psi.Bs[i] = np.tensordot(Gi, np.diag(Sj), axes=[2, 0])
    psi.Ss[j] = Sj
    psi.Bs[j] = Bj
    return log_norm, trunc_err


def accum_norm(psi, U_bonds, chi_max, eps, current_log_norm=0.0):
    """Run one TEBD step [even, odd, even] and accumulate the log-norm.

    Parameters
    ----------
    current_log_norm : float
        Running total log-norm from all previous steps (start at 0.0).

    Returns
    -------
    new_log_norm  : float -- updated accumulated log-norm
    max_trunc_err : float -- max truncation error over all bonds in this step
    """
    Nbonds = psi.L - 1 if psi.bc == 'finite' else psi.L
    log_norm = current_log_norm
    max_trunc_err = 0.0
    for k in [0, 1, 0]:   # even, odd, even
        for i_bond in range(k, Nbonds, 2):
            ln, te = update_bond(psi, i_bond, U_bonds[i_bond], chi_max, eps)
            log_norm += ln
            max_trunc_err = max(max_trunc_err, te)
    return log_norm, max_trunc_err

## Observable Measurement

### Identity bra vector

The **identity bra** $\langle\langle I|$ is the vectorized identity operator. At each site, the local identity vector has components:
$$
(I_{\text{vec}})_\alpha = \delta_{m,n} \quad \text{where} \quad \alpha = m \cdot d + n
$$
i.e., it is non-zero only for the diagonal entries $\alpha \in \{0, 3\}$ (for $d=2$).

### Expectation values

The expectation value of a physical observable $O$ is:
$$
\text{Tr}(O \rho) = \frac{\langle\langle I | O \otimes I | \tilde{\rho} \rangle\rangle}{\langle\langle I | \tilde{\rho} \rangle\rangle}
$$
where:
- $O \otimes I$ is the super-operator that acts on the ket space only
- The denominator $\langle\langle I | \tilde{\rho} \rangle\rangle$ corrects for the Frobenius normalization of $|\tilde{\rho}\rangle\rangle$
- The contraction is performed as a left-to-right transfer matrix sweep

The `measure_phys(psi, op, d)` function implements this for a single-site physical operator `op` at every site, returning a vector of expectation values of length $L$.

In [ ]:
# Observable measurement functions (from TFIM_tebd_open.py)

def _identity_vec(d):
    """Local identity vector: I_vec[alpha] = delta_{m,n} for alpha = m*d + n."""
    ds = d * d
    v = np.zeros(ds)
    for n in range(d):
        v[n * d + n] = 1.0
    return v


def _contract_identity(psi, I_vec, op_site=None):
    """Compute <<I| [op at site i] |rho~>> via left-to-right transfer matrices.

    op_site : None      -> compute <<I|rho~>>
              (i, M)   -> insert M (ds x ds) at site i, then compute <<I|M|rho~>>
    """
    T = np.ones((1, 1), dtype=complex)
    for k in range(psi.L):
        B = psi.Bs[k]                    # (chi_L, ds, chi_R)
        if op_site is not None and k == op_site[0]:
            M  = op_site[1]
            MB = np.tensordot(M, B, axes=([1], [1]))   # ds chi_L chi_R
            MB = MB.transpose(1, 0, 2)                 # chi_L ds chi_R
            T  = np.einsum('ij,a,jal->il', T, I_vec, MB)
        else:
            T = np.einsum('ij,a,jal->il', T, I_vec, B)
    return T[0, 0]


def measure_phys(psi, op_phys, d):
    """Compute Tr(O rho) at every site using <<I|O x I|rho~>> / <<I|rho~>>.

    op_phys : (d, d) single-site physical operator.
    Returns np.ndarray of shape (L,).
    """
    I_vec  = _identity_vec(d)
    O_super = np.kron(op_phys, np.eye(d))       # (ds, ds)
    denom   = _contract_identity(psi, I_vec)
    return np.array([
        np.real(_contract_identity(psi, I_vec, op_site=(i, O_super)) / denom)
        for i in range(psi.L)
    ])


def frobenius_norm(psi, d):
    """Return ||rho||_F = 1 / <<I|rho~>>.

    TEBD keeps ||rho~||_F = 1, so the physical density matrix has
    rho = c * rho~ with c = ||rho||_F. For a pure state c=1; dephasing
    drives c toward 1/sqrt(d) (maximally mixed state).
    Tr(rho) = 1 is always conserved; the observable formula divides by
    <<I|rho~>> = 1/c to compensate.
    """
    return 1.0 / np.real(_contract_identity(psi, _identity_vec(d)))


# Verify: for the initial |+x><+x| state, <sigma^x> should be 1, <sigma^z> should be 0
d = 2
sz = np.array([[1., 0.], [0., -1.]])
sx = np.array([[0., 1.], [1., 0.]])

psi_test = init_purification_MPS(L=4, d=d, bc='finite', state='+x')
print("Initial <sigma^x>:", measure_phys(psi_test, sx, d))
print("Initial <sigma^z>:", measure_phys(psi_test, sz, d))
print("Initial Frobenius norm:", frobenius_norm(psi_test, d))

## Running the Open-System Simulation

We simulate the open TFIM with parameters:
- $L = 6$ sites
- $J = 1.0$ (Ising coupling), $B = 0.5$ (transverse field)
- $\gamma = 0.3$ (dephasing rate for $L_i = \sigma^z_i$)
- $t_{\max} = 4.0$, $\Delta t = 0.01$
- Initial state: $\rho_0 = |{+x}\rangle\langle{+x}|^{\otimes L}$

The simulation calls `TFIM_TEBD_open` from `TFIM_tebd_open.py` and records:
- $\langle\sigma^z_i\rangle(t)$ and $\langle\sigma^x_i\rangle(t)$ (mean magnetization)
- Entanglement entropy per bond
- Frobenius norm $\|\rho\|_F$ and trace $\text{Tr}(\rho)$ (sanity checks)
- Truncation error

In [ ]:
# Run the full open-system simulation using TFIM_TEBD_open from TFIM_tebd_open.py
from TFIM_tebd_open import TFIM_TEBD_open, _identity_vec, _contract_identity

L     = 6
J     = 1.0
B     = 0.5
gamma = 0.3
tmax  = 4.0
dt    = 0.01

tspan, Sz_t, Sx_t, S_ent_t, trunc_t = TFIM_TEBD_open(
    L=L, J=J, B=B, gamma=gamma, tmax=tmax, dt=dt,
    chi_max=50, eps=1e-8, init_state='+x'
)

print("Simulation complete.")
print("Output arrays:")
print("  tspan shape:", tspan.shape)
print("  Sz_t  shape:", Sz_t.shape,  "  (Nsteps, L)")
print("  Sx_t  shape:", Sx_t.shape,  "  (Nsteps, L)")
print("  S_ent shape:", S_ent_t.shape, "  (Nsteps, L-1)")

In [ ]:
# TFIM_TEBD_open function definition (from TFIM_tebd_open.py)
# Reproduced here for reference — the actual simulation above uses the imported version.

def TFIM_TEBD_open_inline(L, J, B, gamma, tmax, dt,
                           chi_max=50, eps=1e-8, init_state='+x'):
    """Simulate the open TFIM with dephasing noise using vectorized-MPS TEBD.

    Hamiltonian : H = -J sum sigma^x_i sigma^x_{i+1} + B sum sigma^z_i
    Jump operator: L_i = sigma^z_i  (local dephasing)
    Off-diagonal elements decay as rho_{01}(t) = rho_{01}(0) * exp(-2*gamma*t).

    Parameters
    ----------
    L          : number of physical sites
    J, B       : Ising coupling and transverse field
    gamma      : dephasing rate
    tmax, dt   : total time and time step
    init_state : '+x' -> rho = |+x><+x|^L  (default)
                 'up' -> rho = |0><0|^L
    """
    import a_mps
    import b_model

    d = 2
    Nsteps = int(tmax / dt)
    tspan  = np.linspace(0.0, tmax, Nsteps)

    sz = np.array([[1., 0.], [0., -1.]])
    sx = np.array([[0., 1.], [1., 0.]])

    M   = b_model.TFIModelOpen(L=L, J=J, B=B,
                                jump_ops=[sz], gammas=[gamma], bc='finite')
    psi = a_mps.init_purification_MPS(L, d, bc='finite', state=init_state)

    U_bonds = calc_U_bonds_open(M.K_bonds, dt)

    Sz_t, Sx_t, S_ent_t, frob_t, trace_t, trunc_t = [], [], [], [], [], []
    log_norm = 0.0

    for n in range(Nsteps):
        if n % max(1, Nsteps // 10) == 0:
            print(f"t = {tspan[n]:.2f},  bond dims = {psi.get_chi()}")

        log_norm, max_trunc = accum_norm(psi, U_bonds, chi_max, eps, log_norm)

        Sz_t.append(measure_phys(psi, sz, d))
        Sx_t.append(measure_phys(psi, sx, d))
        S_ent_t.append(psi.entanglement_entropy())
        frob_t.append(frobenius_norm(psi, d))
        bra_I = np.real(_contract_identity(psi, _identity_vec(d)))
        trace_t.append(np.exp(log_norm) * bra_I)
        trunc_t.append(max_trunc)

    return (tspan,
            np.array(Sz_t), np.array(Sx_t),
            np.array(S_ent_t), np.array(trunc_t))

print("Function TFIM_TEBD_open_inline defined (for reference).")
print("The simulation above used the optimized version from TFIM_tebd_open.py.")

## Validation

### 1. Dephasing decay: $\langle\sigma^x\rangle \propto e^{-2\gamma t}$

For **pure dephasing** ($L = \sigma^z$, $J = 0$), the Lindblad equation gives an exact analytical solution. The off-diagonal elements of $\rho$ decay as:
$$
\rho_{01}(t) = \rho_{01}(0)\, e^{-2\gamma t}
$$
Since $\langle\sigma^x\rangle = \rho_{01} + \rho_{10} = 2\text{Re}(\rho_{01})$, we expect:
$$
\langle\sigma^x\rangle(t) = \langle\sigma^x\rangle(0)\, e^{-2\gamma t}
$$
With the Ising coupling $J > 0$, this exact result is modified, but the dephasing still dominates at large $\gamma t$. The `_plot` function inside `TFIM_TEBD_open` overlays $e^{-2\gamma t}$ for comparison.

### 2. Trace conservation: $\text{Tr}(\rho) = 1$

The Lindblad equation preserves the trace of $\rho$. In the MPS simulation, we track:
$$
\text{Tr}(\rho(t)) = e^{\log\mathcal{N}(t)} \cdot \langle\langle I | \tilde{\rho}(t) \rangle\rangle
$$
This should remain equal to 1 throughout the simulation (up to truncation error).

### 3. Frobenius norm decay

The Frobenius norm $\|\rho\|_F = \sqrt{\text{Tr}(\rho^2)}$ is the **purity**. For a pure state, $\|\rho\|_F = 1$; for the maximally mixed state ($d=2$), $\|\rho\|_F = 1/\sqrt{2}$. Dephasing drives the system from pure to mixed, so $\|\rho\|_F$ should decrease monotonically from 1 toward $1/\sqrt{2}$.

### Summary of expected behaviors

| Observable | $t=0$ | $t \to \infty$ |
|-----------|-------|----------------|
| $\langle\sigma^x\rangle$ | 1 | 0 (dephasing kills coherence) |
| $\langle\sigma^z\rangle$ | 0 | oscillates then decays |
| $\|\rho\|_F$ | 1 | $1/\sqrt{2}$ |
| $\text{Tr}(\rho)$ | 1 | 1 (conserved) |
| Bond entropy | 0 | grows then saturates |